Toy Monte Carlo simulation for studying phase transitions in the theory of a single-component real scalar field.

The theory has potential

$$ V(\phi) = \sigma_3 \phi + \frac{1}{2} m^2 \phi^2 + \frac{1}{3!} g_3 \phi^3 + \frac{1}{4!} \lambda \phi^4 $$

It has been studied in papers including:
- Oliver Gould, **Real scalar phase transitions: a nonperturbative analysis**, https://arxiv.org/abs/2101.05528
- Oliver Gould, Annd Kormu and David J. Weir **A nonperturbative test of nucleation calculations for strong phase transitions**, https://arxiv.org/abs/2404.01876

The code used in those papers (`scalnuc`) is a good reference for a C implementation of a 'full-scale' simulation of this theory: https://bitbucket.org/og113/scalnuc with all the 'extras' I mention at the end of this worksheet. The overrelaxation implementation below is borrowed from `scalnuc`.

As a toy implementation, the code below is not highly optimised, and lacks some of the needed features (e.g. the multicanonical algorithm) for running useful simulations. However, it should be perfectly possible to rewrite and extend this in another language - or even do some serious optimising in python - and get something that scales to similar lattice sizes as in the above papers.

In [ ]:
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt
import scipy.optimize
import copy

# Needed for animations
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Needed for interactive slider bars
import ipywidgets

In [ ]:
@dataclass
class Parameters:
    # Number of lattice sites in each direction
    Nx: int
    Ny: int
    Nz: int

    # Lattice spacing
    a: float

    # Bare, lattice potential parameters
    sigma3: float
    m3sq: float
    g3: float
    lambda3: float

    # Random number generator, needs initialised with a suitable seed
    rng: np.random.Generator

Setting up the theory - what is the potential, the Laplacian, etc.

In [ ]:
# The potential at a single lattice site
def local_potential(phi: float, p: Parameters):
    return (
        p.sigma3 * phi
        + (1.0 / 2.0) * p.m3sq * phi**2
        + (1.0 / 6.0) * p.g3 * phi**3
        + (1.0 / 24.0) * p.lambda3 * phi**4
    )


# The discretised Laplacian at a given site (x, y, z) - just a simple second order central difference
def laplacian(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += lattice[(x + 1) % p.Nx][y][z]
    total += lattice[x][(y + 1) % p.Ny][z]
    total += lattice[x][y][(z + 1) % p.Nz]
    total += lattice[(x + p.Nx - 1) % p.Nx][y][z]
    total += lattice[x][(y + p.Ny - 1) % p.Ny][z]
    total += lattice[x][y][(z + p.Nz - 1) % p.Nz]

    total -= 6.0 * lattice[x][y][z]

    return total / p.a**2


# This is the contribution to the action at a site (x, y, z), due to the Laplacian terms both at that site and neighbouring sites.
def laplacian_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += 2.0 * lattice[(x + 1) % p.Nx][y][z]
    total += 2.0 * lattice[x][(y + 1) % p.Ny][z]
    total += 2.0 * lattice[x][y][(z + 1) % p.Nz]
    total += 2.0 * lattice[(x + p.Nx - 1) % p.Nx][y][z]
    total += 2.0 * lattice[x][(y + p.Ny - 1) % p.Ny][z]
    total += 2.0 * lattice[x][y][(z + p.Nz - 1) % p.Nz]

    total -= 6.0 * lattice[x][y][z]

    return total / p.a**2


# This is the total contribution to the action from all terms depending on the value of the field at a site (x, y, z)
# - we use this when e.g. updating the site with the Metropolis algorithm, to quickly evaluate the change in the action
def lattice_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += (
        (-1.0 / 2.0) * lattice[x][y][z] * laplacian_contribution(lattice, x, y, z, p)
    )
    total += local_potential(lattice[x][y][z], p)

    return total


# This is the total potential energy for the system
def potential(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += local_potential(lattice[x][y][z], p)

    return total


# And this is the total lattice action for the system
# Notice the 'integrated-by-parts' gradient term using the above Laplacian function
def lattice_action(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += (
                    (-1.0 / 2.0) * lattice[x][y][z] * laplacian(lattice, x, y, z, p)
                )
                total += local_potential(lattice[x][y][z], p)

    return total

Some ways of setting up our lattice:
- 'cold' where everything is set to a constant (zero),
- and 'warm' where we use some (uniform) random numbers everywhere

Neither are particularly likely samples from the canonical ensemble, but a good Monte Carlo algorithm should quickly converge.

In [ ]:
def init_lattice_cold(p: Parameters):
    lattice = np.zeros(shape=(p.Nx, p.Ny, p.Nz), dtype=float)
    return lattice


def init_lattice_warm(p: Parameters, seed=1):

    rng = np.random.default_rng(seed=seed)
    lattice = rng.uniform(-1, 1, size=(p.Nx, p.Ny, p.Nz))
    return lattice

Do importance sampling with a simple Metropolis update for a site (x, y, z):

- propose a new value for the site
- evaluate the change in the action $\Delta S = S_\text{after} - S_\text{before}$ (which depends only on nearest neighbours, evaluated by `lattice_contribution()`)
- accept the change with probability $W = \text{min}(1, e^{-\Delta S})$.
  - in other words, if it reduces the action ($\Delta S < 0$), always accept the change
  - if it increases the action, ($\Delta S > 0$), accept the change with probability $e^{-\Delta S}$; we can do this by generating a uniform random number between 0 and 1 and comparing the two.

If we apply this to all sites in the lattice systematically, then this satisfies the 'detailed balance' and 'strong ergodicity' requirements of a Monte Carlo algorithm. And it will generate configurations of the lattice with relative probabilities given by the canonical ensemble, making averages of obserables direct averages of the time series.

In [ ]:
# Returns 0 or 1 and the change in the action due to the update.
def metropolis_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    original = lattice[x][y][z]

    before = lattice_contribution(lattice, x, y, z, p)

    lattice[x][y][z] += p.rng.uniform(-1, 1)

    after = lattice_contribution(lattice, x, y, z, p)

    if after < before or np.exp(-1.0 * (after - before)) > p.rng.uniform(0, 1):
        return 1, (after - before)

    lattice[x][y][z] = original

    return 0, 0.0

Overrelaxation algorithm, borrowed from scalnuc (https://bitbucket.org/og113/scalnuc). This is included in case it helps speed up the simulations a bit, given the limited resources we have available running in a Jupyter notebook.

The scalnuc implementation is in turn based on the discussion in the paper https://arxiv.org/abs/hep-lat/9510020 (equation 3.1 onwards) which in turn comes from e.g. https://arxiv.org/abs/hep-lat/9403024 section 2.

In [ ]:
def overrelax_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):

    # First step, find a value of the field for which the 'local' action is unchanged.

    phi_0 = lattice[x][y][z]

    alpha = (1.0 / 24.0) * p.lambda3
    beta = (1.0 / 6.0) * p.g3
    gamma = (-1.0 / 2.0) * -6.0 + (1.0 / 2.0) * p.m3sq
    delta = (
        2.0 * (-1.0 / 2.0) * (laplacian(lattice, x, y, z, p) + 6.0 * phi_0) + p.sigma3
    )

    #   this is equivalent to solving
    #        alpha*(phi-phi0)*(phi**3 + a*phi**2 + b*phi + d) = 0
    #    where
    #        alpha = phi0
    #        beta = b/a + phi0**2
    #        gamma = d/a + alpha*beta

    a = phi_0 + beta / alpha
    b = phi_0 * a + gamma / alpha
    d = phi_0 * b + delta / alpha

    #    solving
    #        phi**3 + a*phi**2 + b*phi + d = 0

    def poly(x):
        return x**3 + a * x**2 + b * x + d

    roots = scipy.optimize.fsolve(poly, 0.1)

    phi = roots[0]

    if len(roots) == 3:
        prob = p.rng.uniform(0, 1)

        if prob > 2.0 / 3.0:
            phi = roots[2]
        elif prob > 1.0 / 3.0:
            phi = roots[1]

    # Second step, since the measure has changed under the mapping to the new phi, compute the change in the measure
    measure = (
        phi_0 * (4.0 * alpha * phi_0 * phi_0 + 3.0 * beta * phi_0 + 2.0 * gamma) + delta
    )
    measure /= phi * (4.0 * alpha * phi * phi + 3.0 * beta * phi + 2.0 * gamma) + delta
    measure = np.log(np.abs(measure))

    lattice[x][y][z] = phi

    # Do an accept-reject step on the new measure
    if measure > np.log(1.0) or measure > np.log(p.rng.uniform(0, 1)):
        return 1

    lattice[x][y][z] = phi_0

    return 0

A 'sweep' refers to visiting every site in order. Note that we do a 'checkerboard' update where we update all the 'odd' (black chess board squares) sites and then the 'even' (white chess board squares) sites. That way we avoid updating sites which depend on one another sequentially, and all updates are indepdenent.

It would also let us parallelise the update if our code was set up for that.

In [ ]:
# Perform a single sweep of metropolis updates
# Returns the number of sites updated, and the number of updates that were accepted.
def sweep(lattice: np.ndarray, p: Parameters):
    total_change = 0.0

    updated = 0
    accepted = 0

    # before = lattice_action(lattice, p)

    for evenodd in (0, 1):
        for x in range(p.Nx):
            for y in range(p.Ny):
                for z in range(p.Nz):
                    if (x + y + z) % 2 == evenodd:
                        accept, change = metropolis_update(lattice, x, y, z, p)
                        total_change += change
                        updated += 1
                        accepted += accept

    # after = lattice_action(lattice, p)

    return updated, accepted


# The same, but for overrelaxation.
def sweep_overrelax(lattice: np.ndarray, p: Parameters):
    total_change = 0.0

    updated = 0
    accepted = 0

    for evenodd in (0, 1):
        for x in range(p.Nx):
            for y in range(p.Ny):
                for z in range(p.Nz):
                    if (x + y + z) % 2 == evenodd:
                        accept = overrelax_update(lattice, x, y, z, p)
                        updated += 1
                        accepted += accept

    after = lattice_action(lattice, p)
    # print(before, after, total_change, after-before, (after-before)-total_change)
    return updated, accepted

Observables/measurables. These include volume-average monomials $\overline{\phi}$, $\overline{\phi^2}$ and $\overline{\phi^4}$.

In [ ]:
def avg_phi(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z]

    return total / (p.Nx * p.Ny * p.Nz)


def avg_phi_sq(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z] ** 2

    return total / (p.Nx * p.Ny * p.Nz)


def avg_phi_4(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z] ** 4

    return total / (p.Nx * p.Ny * p.Nz)

Run the simulation for a number of iterations. One can set the number of Metropolis sweeps (`n_metropolis`) and overelaxation sweeps (`n_overrelax`) per iteration. One can also save 2D slices through the lattice to look at.

In [ ]:
# Run the simulation.
# Returns a list of measurements, consisting of a series of tuples of volume-averaged (phi, phi^2, phi^4, action, potential).
# If save_slices is True, then also store and return a list of slices.
def run(
    iterations: int,
    lattice: np.ndarray,
    p: Parameters,
    save_slices=False,
    n_metropolis=8,
    n_overrelax=1,
):

    total_updated = 0
    total_accepted = 0

    total_overrelax_updated = 0
    total_overrelax_accepted = 0
    results = []
    if save_slices:
        slices = []

    for i in range(iterations):

        # Perform the requested number of Metropolis and overrelax sweeps each iteration

        for j in range(n_metropolis):
            updated, accepted = sweep(lattice, p)
            total_updated += updated
            total_accepted += accepted

        for j in range(n_overrelax):
            updated, accepted = sweep_overrelax(lattice, p)
            total_overrelax_updated += updated
            total_overrelax_accepted += accepted

        # After each iteration, save the monomials, the total action, and the potential
        # This probably does not need to be done every iteration in a real simulation
        results.append(
            (
                avg_phi(lattice, p),
                avg_phi_sq(lattice, p),
                avg_phi_4(lattice, p),
                lattice_action(lattice, p),
                potential(lattice, p),
            )
        )

        # Store a copy of a slice through the lattice in case we want to visualise it
        if save_slices:
            slices.append(copy.deepcopy(lattice[0, :]))

    # Print the acceptance rates
    print(
        "Metropolis: %d sites updated, %d accepted, %g rate"
        % (total_updated, total_accepted, float(total_accepted) / float(total_updated))
    )

    if total_overrelax_updated > 0:
        print(
            "Overrelax: %d sites updated, %d accepted, %g rate"
            % (
                total_overrelax_updated,
                total_overrelax_accepted,
                float(total_overrelax_accepted) / float(total_overrelax_updated),
            )
        )

    if save_slices:
        return results, slices

    return results

These parameters produce a very strong phase transition. It is manageable to run it at 5^3 in the notebook.

In [ ]:
p = Parameters(
    Nx=5,
    Ny=5,
    Nz=5,
    a=1,
    sigma3=0.095,
    m3sq=-0.04,
    g3=-0.6,
    lambda3=1,
    rng=np.random.default_rng(seed=1),
)

lattice = init_lattice_warm(p, seed=10)

Plot the corresponding bare potential.

In [ ]:
x = np.linspace(-1.5, 2.5, 50)
plt.plot(x, local_potential(x, p))
plt.xlabel(r"$\phi$")
plt.ylabel(r"$V(\phi)$")

In [ ]:
results_all = []
slices_all = []

Do the simulation - **can be slow**.

Running this many iterations takes a minute on my laptop (M4 MacBook Air), and about two minutes on mybinder.org:

In [ ]:
results, slices = run(5000, lattice, p, save_slices=True)

results_all += results
slices_all += slices

Plot time series of the various quantities recorded during the simulation.

In [ ]:
phi = np.array(results_all)[:, 0]
phisq = np.array(results_all)[:, 1]
phi4 = np.array(results_all)[:, 2]
action = np.array(results_all)[:, 3]
pot = np.array(results_all)[:, 4] / (p.Nx * p.Ny * p.Nz)

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(figsize=(12, 3), ncols=5)

ax1.plot(range(len(phi)), phi)
ax1.set_xlabel("measurement")
ax1.set_ylabel(r"$\overline{\phi}$")
ax2.plot(range(len(phisq)), phisq)
ax2.set_xlabel("measurement")
ax2.set_ylabel(r"$\overline{\phi^2}$")
ax3.plot(range(len(phi4)), phi4)
ax3.set_xlabel("measurement")
ax3.set_ylabel(r"$\overline{\phi^4}$")
ax4.plot(range(len(action)), action)
ax4.set_xlabel("measurement")
ax4.set_ylabel(r"$S$")
ax5.plot(range(len(pot)), pot)
ax5.set_xlabel("measurement")
ax5.set_ylabel(r"$\overline{V(\phi)}$")

plt.tight_layout()
plt.show()

Question: which quantities above can distinguish between the two phases, and thus function as order parameters?

Anyway - making this movie below takes about fifteen seconds on my MacBook Air M4. It shows (at left) the potential the where the volume-averaged value of $\phi$ is currently, and (at right) a slice through the lattice simulation.

In [ ]:
def plot_slices():
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 3.5))

    x = np.linspace(-1.5, 2.5, 50)

    # Initial state, needed for colorbar
    pot_plot = ax1.plot(x, local_potential(x, p))
    ymin = ax1.get_ylim()[0]
    ax1.set_xlabel(r"$\phi$ or $\overline{\phi}$")
    ax1.set_ylabel(r"$V(\phi)$")
    sc_plot = ax1.scatter(phi[0], ymin)
    lattice_slice = ax2.imshow(slices_all[0], vmin=-1.5, vmax=2.5)
    cbar = fig.colorbar(lattice_slice, ax=ax2, shrink=1.0)
    cbar.set_label(r"$\phi(\mathbf{x})$")
    plt.tight_layout()

    def update(frame):
        ax1.clear()
        ax2.clear()
        pot_plot = ax1.plot(x, local_potential(x, p))
        ax1.set_xlabel(r"$\phi$ or $\overline{\phi}$")
        ax1.set_ylabel(r"$V(\phi)$")
        sc_plot = ax1.scatter(phi[frame], ymin)
        lattice_slice = ax2.imshow(slices_all[frame], vmin=-1.5, vmax=2.5)

    ani = FuncAnimation(
        fig, update, frames=range(0, len(slices_all), 20), interval=100, repeat=False
    )
    return ani


movie = plot_slices().to_jshtml()
plt.close()
HTML(movie)

Now, plot a histogram of phi, showing that it has a double-peaked structure, corresponding to the two phases. The minimum between the two is much lower - maybe a tenth of the peaks - but not so small as to make tunnelling impossible. However, it will go down exponentially with volume, making simple Monte Carlo simulations like this very difficult as we crank up the volume.

In [ ]:
(counts, edges, patches) = plt.hist(phi, bins=50)
plt.xlabel(r"$\overline{\phi}$")
plt.ylabel(r"Frequency")

On the other hand, the action *cannot* distinguish between the two phases.

In [ ]:
(counts, edges, patches) = plt.hist(action, bins=50)
plt.xlabel(r"$S$")
plt.ylabel(r"Frequency")

Reweight the simulation to a different parameter choice, based on monomial terms in the action that we recorded earlier.

In [ ]:
def reweight(dsigma=0, dmsq=0, dlambda=0):
    numerator = np.exp(
        -(dsigma * phi + (1.0 / 2.0) * dmsq * phisq + (1.0 / 24.0) * dlambda * phi4)
    )
    denominator = np.sum(numerator) / len(results_all)
    weights = numerator / denominator
    return weights

We can use this to look at the jump in the 'condensate', $\langle\phi\rangle$, but it will only get truly sharp in the continuum limit.

In [ ]:
delta = np.arange(-10, 10, 0.01)
reweighted = []
for d in delta:
    reweighted.append(np.mean(phi * reweight(dmsq=d)))
plt.plot(delta, reweighted)
plt.xlabel(r"$\Delta = m^2 - m_0^2$")
plt.ylabel(r"$\langle \overline{\phi} \rangle$")
plt.plot()

Here you can interactively experiment with reweighting the simulation data. How far would you trust them?

In [ ]:
def plot_reweight(dsigma=0, dmsq=0, dlambda=0):
    weights = reweight(dsigma, dmsq, dlambda)
    plt.hist(phi, bins=50, weights=weights)
    plt.ylim(0, 1000)
    plt.show()


ipywidgets.interact(
    plot_reweight, dsigma=(-7.0, 7.0), dmsq=(-7.0, 7.0), dlambda=(-7.0, 7.0)
)

Things to try:

- **Easy**
  - Run for longer, and see what the peaks stabilise out at.
  - Decrease the lattice volume to say 4x4x4, and try increasing (if you have the time and patience) to 6x6x6). How often does the simulation tunnel back and forth? How long do you need to run it for?
  - The overrelaxation is computationally costly. Is it worth it? Study how it works.

- **Medium**
  - Study autocorrelation times. How autocorrelated are these quantities. Do all the measurables have the same autocorrelation time?
  - Consider statistical errors. How many measurements does one have if one only considers say every N autocorrelation times an independent measurement? What about statistical bias - can one use jackknife or bootstrap?
  - One can use an 'improved' Laplacian, say the fourth order accurate, second finite difference (see e.g. https://en.wikipedia.org/wiki/Finite_difference_coefficient). Does it make a difference to our results?

- **Hard** [these will need you to run the code for a while, probably on your own laptop, or do some extended thinking and calculating]
  - Add support for the multicanonical algorithm, and use it to build a weight function. The 'perfect' weight function tells you the free energy of the system immediately.
  - Implement the lattice-continuum relations described in https://arxiv.org/abs/2101.05528 and run the simulation to reproduce e.g. one of the curves in Figure 5.
  - Reproduce the matching of the parameters of the 4D theory to the 3D theory.